In [ ]:
import os
import pandas as pd

In [ ]:
label_counts = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, "total": 0}

In [ ]:
exclude_theoretical = True

# Open llm_results.csv
csv_file_path = "../../results/llm_results.csv"

if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"CSV file not found: {csv_file_path}")

df = pd.read_csv(csv_file_path)

In [ ]:
len(df)

In [ ]:
if exclude_theoretical:
    # remove all of the rows where research_type is 1
    df = df[df["research_type"] != 1]

In [ ]:
len(df)

In [ ]:
label_results = {}

label_results = {
    2014: label_counts.copy(),
    2015: label_counts.copy(),
    2016: label_counts.copy(),
    2017: label_counts.copy(),
    2018: label_counts.copy(),
    2019: label_counts.copy(),
    2020: label_counts.copy(),
    2021: label_counts.copy(),
    2022: label_counts.copy(),
    2023: label_counts.copy(),
    2024: label_counts.copy(),
}

In [ ]:
df.head()

In [ ]:
for _, row in df.iterrows():
    year = row["year"]

    paper_total = int(
        row["pseudocode"]
        + row["open_source_code"]
        + row["train"]
        + row["validation"]
        + row["hardware_specification"]
        + row["software_dependencies"]
        + row["experiment_setup"]
    )

    label_results[year][paper_total] += 1

    label_results[year]["total"] += 1

In [ ]:
# Convert label_results to percentages
for year in label_results:
    for label in label_results[year]:
        label_results[year][label] = round(
            (label_results[year][label] / label_results[year]["total"]), 3
        )

    del label_results[year]["total"]

In [ ]:
# Make sure the sum of the rates is 1 for each year
for year in label_results:

    total = 0

    for label in label_results[year]:
        total += label_results[year][label]

    print(total)

In [ ]:
from matplotlib.ticker import FuncFormatter

font_size = 18

# Convert label_results to a DataFrame
df = pd.DataFrame(label_results).T

# Reorder the columns in the desired order
df = df[
    [
        7,
        6,
        5,
        4,
        3,
        2,
        1,
        0,
    ]
]

# Plot the stacked area chart
ax = df.plot(
    kind="area",
    stacked=True,
    figsize=(12, 7),
    color={
        7: "#f3b23e", # Gold
        6: "#e37137", # Orange
        5: "#d62728", # Red
        4: "#e662bd", # Pink
        3: "#5dbaf7", # Light Blue
        2: "#9467bd", # Purple
        1: "#8c564b", # Brown
        0: "#2ca02c", # Green
    },
)

# Set x-axis limits to include 2014 and 2024 flush with the y-axis
ax.set_xlim(2014, 2024)
ax.set_ylim(0, 1)

# Convert y-axis to percentage
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y * 100)}%"))

# Set x-axis ticks for each year
ax.set_xticks([year for year in range(2014, 2025)])

# Set y-axis ticks every 10%
ax.set_yticks([i / 10 for i in range(0, 11)])

# Add a secondary y-axis on the right
ax_right = ax.twinx()
ax_right.set_ylim(0, 1)

# Convert the right y-axis to percentage as well
ax_right.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y * 100)}%"))

# Set y-axis ticks every 10% for the right axis
ax_right.set_yticks([i / 10 for i in range(0, 11)])

# Increase font size for all text in the plot
ax.tick_params(axis="both", which="major", labelsize=font_size - 4)
ax_right.tick_params(axis="both", which="major", labelsize=font_size - 4)

# Set the title and labels
ax.set_title("Percentage of Papers by Number of Documented Reproducibility Variables", fontsize=font_size)
ax.set_ylabel("Percentage of Empirical Papers", fontsize=font_size - 2)
ax.set_xlabel("Year", fontsize=font_size - 2)

ax.set_prop_cycle(None)  # Reset color cycle for legend mapping

handles, labels = ax.get_legend_handles_labels()
ax.legend(
    title="Number of Documented Reproducibility Variables in the Paper",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=8,
    fontsize=font_size - 4,
    title_fontsize=font_size - 2,
)

ax.figure.savefig(
    "../../plots/num_reproducibility_variables.pdf", format="pdf", bbox_inches="tight"
)